# 06 — Final controlled comparison and model selection

## Objective

Compare all five matched experiments in one table and select the deployable model
using validation-seen IoU only. Test-seen and test-unseen results describe final
performance but never influence model selection.


In [1]:
from pathlib import Path
import json
import os
import sys

from IPython.display import Image, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "datasets").is_dir() and (path / "final_model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository or final_training_notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_ID = os.environ.get("FINAL_TRAINING_RUN_ID", "manual")
FRESH_TRAINING = True
RESULT_ROOT = PROJECT_ROOT / "training_results_corrected"
POINT_ROOT = RESULT_ROOT
print("Project:", PROJECT_ROOT)
print("Run ID:", RUN_ID)


Project: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation
Run ID: manual


## 1. Comparison and selection implementation

This section validates the five experiment summaries, constructs the complete
comparison table, applies the validation-only selection rule, copies the selected
UI checkpoint, writes the model registry, and saves the primary comparison plot.


In [2]:
import json
import os
import shutil

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

EXPERIMENTS = {'baseline_object_mask': None, 'fixed_uvd': None, 'query_gated_uvd': None, 'rotation_consistent': None, 'geometry_dropout': None}

def run_id():
    return os.environ.get("FINAL_TRAINING_RUN_ID", "manual")

def results_root():
    path = PROJECT_ROOT / "training_results_corrected"
    path.mkdir(parents=True, exist_ok=True)
    return path

def points_root():
    path = PROJECT_ROOT / "training_results_corrected"
    path.mkdir(parents=True, exist_ok=True)
    return path

def build_comparison() -> pd.DataFrame:
    result_base = results_root()
    point_base = points_root()
    missing = [name for name in EXPERIMENTS if not (result_base / name / "summary.csv").is_file()]
    if missing:
        raise FileNotFoundError(f"Training summaries are missing for: {missing}")
    frames = [pd.read_csv(result_base / name / "summary.csv") for name in EXPERIMENTS]
    comparison = pd.concat(frames, ignore_index=True)
    comparison.to_csv(result_base / "all_experiment_comparison.csv", index=False)
    validation = comparison.drop_duplicates("experiment").sort_values("validation_iou", ascending=False)
    selected = str(validation.iloc[0].experiment)
    shutil.copy2(point_base / selected / "ui_model.pt", point_base / "best_model.pt")
    registry = {
        "run_id": run_id(),
        "selection_rule": "maximum validation_seen IoU; no test metric used for selection",
        "selected_model": selected,
        "selected_checkpoint": str((point_base / "best_model.pt").relative_to(PROJECT_ROOT)),
        "models": {
            name: {
                "checkpoint": str((point_base / name / "ui_model.pt").relative_to(PROJECT_ROOT)),
                "validation_iou": float(validation.set_index("experiment").loc[name, "validation_iou"]),
                "selected_epoch": int(validation.set_index("experiment").loc[name, "selected_epoch"]),
            }
            for name in EXPERIMENTS
        },
    }
    (point_base / "model_registry.json").write_text(json.dumps(registry, indent=2) + "\n")
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    validation.sort_values("validation_iou").plot.barh(
        x="experiment", y="validation_iou", legend=False, ax=axes[0], title="Validation checkpoint selection"
    )
    test = comparison.pivot(index="experiment", columns="split", values="iou")
    test.plot.bar(ax=axes[1], title="Final seen and unseen test IoU")
    axes[0].set_xlabel("Validation IoU")
    axes[1].set_ylabel("Mean IoU")
    axes[1].tick_params(axis="x", rotation=30)
    figure.tight_layout()
    figure.savefig(result_base / "final_model_comparison.png", dpi=180, bbox_inches="tight")
    plt.close(figure)
    print(comparison.to_string(index=False))
    print("Selected model:", selected)
    print("Selected UI checkpoint:", point_base / "best_model.pt")
    return comparison


## 2. Ranking, seen/unseen comparison, and deployment choice

The standardized ranking below reports validation IoU, both final test splits,
generalization gaps, changes relative to the object-mask baseline, and leakage.


In [3]:
comparison = build_comparison()
display(comparison.round(4))

validation = comparison.drop_duplicates("experiment").set_index("experiment")
iou = comparison.pivot(index="experiment", columns="split", values="iou")
dice = comparison.pivot(index="experiment", columns="split", values="dice")
leakage = comparison.pivot(index="experiment", columns="split", values="leakage")

ranking = pd.DataFrame(index=validation.index)
ranking["selected epoch"] = validation["selected_epoch"].astype(int)
ranking["validation IoU"] = validation["validation_iou"]
ranking["seen IoU"] = iou["test_seen"]
ranking["unseen IoU"] = iou["test_unseen"]
ranking["seen Dice"] = dice["test_seen"]
ranking["unseen Dice"] = dice["test_unseen"]
ranking["seen leakage"] = leakage["test_seen"]
ranking["unseen leakage"] = leakage["test_unseen"]
ranking["seen−unseen IoU gap"] = ranking["seen IoU"] - ranking["unseen IoU"]

baseline_seen = ranking.loc["baseline_object_mask", "seen IoU"]
baseline_unseen = ranking.loc["baseline_object_mask", "unseen IoU"]
ranking["Δ seen IoU vs baseline"] = ranking["seen IoU"] - baseline_seen
ranking["Δ unseen IoU vs baseline"] = ranking["unseen IoU"] - baseline_unseen
ranking = ranking.sort_values("validation IoU", ascending=False)
display(ranking.round(4))
ranking.to_csv(RESULT_ROOT / "model_ranking.csv")

figure, axes = plt.subplots(1, 3, figsize=(17, 5))
ranking.sort_values("validation IoU")["validation IoU"].plot.barh(
    ax=axes[0], color="#38bdf8"
)
axes[0].set(title="Validation-only selection", xlabel="Validation IoU", ylabel="")

ranking[["seen IoU", "unseen IoU"]].plot.bar(
    ax=axes[1], color=("#818cf8", "#22c55e")
)
axes[1].set(title="Final test IoU", xlabel="", ylabel="Mean IoU", ylim=(0, 1))
axes[1].tick_params(axis="x", rotation=35)

ranking[["Δ seen IoU vs baseline", "Δ unseen IoU vs baseline"]].plot.bar(
    ax=axes[2], color=("#f59e0b", "#ec4899")
)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set(title="Improvement over baseline", xlabel="", ylabel="Δ IoU")
axes[2].tick_params(axis="x", rotation=35)

for axis in axes:
    axis.grid(alpha=0.25)
figure.tight_layout()
dashboard_path = RESULT_ROOT / "submission_model_comparison.png"
figure.savefig(dashboard_path, dpi=180, bbox_inches="tight")
plt.show()

registry = json.loads((POINT_ROOT / "model_registry.json").read_text())
selected = registry["selected_model"]
print("Selection rule:", registry["selection_rule"])
print("Selected model:", selected)
print("Selected validation IoU:", round(float(ranking.loc[selected, "validation IoU"]), 4))
print("Selected checkpoint:", POINT_ROOT / "best_model.pt")
print("Comparison dashboard:", dashboard_path)


          experiment       split  samples      iou     dice  leakage  selected_epoch  validation_iou  validation_dice
baseline_object_mask   test_seen     3371 0.298755 0.398578 0.204555              17        0.284463         0.377728
baseline_object_mask test_unseen     1586 0.221337 0.305721 0.178829              17        0.284463         0.377728
           fixed_uvd   test_seen     3371 0.301558 0.402099 0.197524              15        0.287526         0.382680
           fixed_uvd test_unseen     1586 0.249800 0.339823 0.158164              15        0.287526         0.382680
     query_gated_uvd   test_seen     3371 0.300225 0.399010 0.194519              17        0.288603         0.383943
     query_gated_uvd test_unseen     1586 0.235129 0.323149 0.157877              17        0.288603         0.383943
 rotation_consistent   test_seen     3371 0.308793 0.410565 0.184107              19        0.298125         0.395839
 rotation_consistent test_unseen     1586 0.276057 0.374

,experiment,split,samples,iou,dice,leakage,selected_epoch,validation_iou,validation_dice
0,baseline_object_mask,test_seen,3371,0.2988,0.3986,0.2046,17,0.2845,0.3777
1,baseline_object_mask,test_unseen,1586,0.2213,0.3057,0.1788,17,0.2845,0.3777
2,fixed_uvd,test_seen,3371,0.3016,0.4021,0.1975,15,0.2875,0.3827
3,fixed_uvd,test_unseen,1586,0.2498,0.3398,0.1582,15,0.2875,0.3827
4,query_gated_uvd,test_seen,3371,0.3002,0.3990,0.1945,17,0.2886,0.3839
5,query_gated_uvd,test_unseen,1586,0.2351,0.3231,0.1579,17,0.2886,0.3839
6,rotation_consistent,test_seen,3371,0.3088,0.4106,0.1841,19,0.2981,0.3958
7,rotation_consistent,test_unseen,1586,0.2761,0.3742,0.1521,19,0.2981,0.3958
8,geometry_dropout,test_seen,3371,0.2978,0.3980,0.2122,17,0.2885,0.3832
9,geometry_dropout,test_unseen,1586,0.2384,0.3285,0.1760,17,0.2885,0.3832


,selected epoch,validation IoU,seen IoU,unseen IoU,seen Dice,unseen Dice,seen leakage,unseen leakage,seen−unseen IoU gap,Δ seen IoU vs baseline,Δ unseen IoU vs baseline
experiment,,,,,,,,,,,
rotation_consistent,19,0.2981,0.3088,0.2761,0.4106,0.3742,0.1841,0.1521,0.0327,0.0100,0.0547
query_gated_uvd,17,0.2886,0.3002,0.2351,0.3990,0.3231,0.1945,0.1579,0.0651,0.0015,0.0138
geometry_dropout,17,0.2885,0.2978,0.2384,0.3980,0.3285,0.2122,0.1760,0.0594,-0.0010,0.0171
fixed_uvd,15,0.2875,0.3016,0.2498,0.4021,0.3398,0.1975,0.1582,0.0518,0.0028,0.0285
baseline_object_mask,17,0.2845,0.2988,0.2213,0.3986,0.3057,0.2046,0.1788,0.0774,0.0000,0.0000


Selection rule: maximum validation_seen IoU; no test metric used for selection
Selected model: rotation_consistent
Selected validation IoU: 0.2981
Selected checkpoint: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/training_results/best_model.pt
Comparison dashboard: /home/utn/ojug99ek/Computer_Vision_Object-Relative-Spatial-Encoding-for-Open-Vocabulary-Part-Segmentation/training_results/submission_model_comparison.png


## 3. Conclusion and reporting checklist

The first-ranked model is selected exclusively from validation performance. The
final report should discuss whether its gains are consistent across seen and
unseen classes, whether leakage remains controlled, and whether the improvement
is large enough to justify the added geometry or regularization.

Saved outputs: complete comparison CSV, ranked model CSV, validation/test figures,
selected `best_model.pt`, and `model_registry.json`.